1-Import Libraries

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import re
import warnings
warnings.filterwarnings('ignore')


2- Load DATA

In [40]:
df=pd.read_csv(r"D:\Regression.ml\Project-ML\data\raw\fifa21_raw_data.csv")


3- Show Data

In [41]:
df

,photoUrl,LongName,playerUrl,Nationality,Positions,Name,Age,↓OVA,POT,Team & Contract,...,A/W,D/W,IR,PAC,SHO,PAS,DRI,DEF,PHY,Hits
0,https://cdn.sofifa.com/players/158/023/21_60.png,Lionel Messi,http://sofifa.com/player/158023/lionel-messi/2...,Argentina,RW ST CF,L. Messi,33,93,93,\n\n\n\nFC Barcelona\n2004 ~ 2021\n\n,...,Medium,Low,5 ★,85,92,91,95,38,65,\n372
1,https://cdn.sofifa.com/players/020/801/21_60.png,C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-s...,Portugal,ST LW,Cristiano Ronaldo,35,92,92,\n\n\n\nJuventus\n2018 ~ 2022\n\n,...,High,Low,5 ★,89,93,81,89,35,77,\n344
2,https://cdn.sofifa.com/players/200/389/21_60.png,Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,GK,J. Oblak,27,91,93,\n\n\n\nAtlético Madrid\n2014 ~ 2023\n\n,...,Medium,Medium,3 ★,87,92,78,90,52,90,\n86
3,https://cdn.sofifa.com/players/192/985/21_60.png,Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyn...,Belgium,CAM CM,K. De Bruyne,29,91,91,\n\n\n\nManchester City\n2015 ~ 2023\n\n,...,High,High,4 ★,76,86,93,88,64,78,\n163
4,https://cdn.sofifa.com/players/190/871/21_60.png,Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silv...,Brazil,LW CAM,Neymar Jr,28,91,91,\n\n\n\nParis Saint-Germain\n2017 ~ 2022\n\n,...,High,Medium,5 ★,91,85,86,94,36,59,\n273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18974,https://cdn.sofifa.com/players/257/710/21_60.png,Mengxuan Zhang,http://sofifa.com/player/257710/mengxuan-zhang...,China PR,CB,Zhang Mengxuan,21,47,52,\n\n\n\nChongqing Dangdai Lifan FC SWM Team\n2...,...,Low,Low,1 ★,58,23,26,27,50,48,2
18975,https://cdn.sofifa.com/players/258/736/21_60.png,Vani Da Silva,http://sofifa.com/player/258736/vani-da-silva/...,England,ST,V. Da Silva,17,47,67,\n\n\n\nOldham Athletic\n2020 ~ 2021\n\n,...,Medium,Medium,1 ★,70,46,40,53,16,40,3
18976,https://cdn.sofifa.com/players/247/223/21_60.png,Ao Xia,http://sofifa.com/player/247223/ao-xia/210005/,China PR,CB,Xia Ao,21,47,55,\n\n\n\nWuhan Zall\n2018 ~ 2022\n\n,...,Medium,Medium,1 ★,64,28,26,38,48,51,3
18977,https://cdn.sofifa.com/players/258/760/21_60.png,Ben Hough,http://sofifa.com/player/258760/ben-hough/210005/,England,CM,B. Hough,17,47,67,\n\n\n\nOldham Athletic\n2020 ~ 2021\n\n,...,Medium,Medium,1 ★,64,40,48,49,35,45,5


4- Data overview

In [42]:
print(f"Shape: {df.shape}")

Shape: (18979, 77)


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18979 entries, 0 to 18978
Data columns (total 77 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   photoUrl          18979 non-null  object
 1   LongName          18979 non-null  object
 2   playerUrl         18979 non-null  object
 3   Nationality       18979 non-null  object
 4   Positions         18979 non-null  object
 5   Name              18979 non-null  object
 6   Age               18979 non-null  int64 
 7   ↓OVA              18979 non-null  int64 
 8   POT               18979 non-null  int64 
 9   Team & Contract   18979 non-null  object
 10  ID                18979 non-null  int64 
 11  Height            18979 non-null  object
 12  Weight            18979 non-null  object
 13  foot              18979 non-null  object
 14  BOV               18979 non-null  int64 
 15  BP                18979 non-null  object
 16  Growth            18979 non-null  int64 
 17  Joined      

In [44]:
# Make a copy 
df_clean = df.copy()

5- Remove any '↓' symbols from column names and strip extra whitespace

In [45]:
df_clean.columns = df_clean.columns.str.replace('↓', '').str.strip()

6- Drop unnecessary columns

In [46]:
df_clean.drop(columns=['photoUrl', 'playerUrl', 'LongName'], inplace=True)

7- Hight Cleaning

In [47]:
def parse_height(h):
    try:
        if pd.isna(h): return np.nan
        if isinstance(h, (int, float)): return float(h)
        if "'" in str(h):
            feet, inches = str(h).replace('"', '').split("'")
            return round(int(feet) * 30.48 + int(inches) * 2.54, 1)
        return float(h)
    except: return np.nan

df_clean['Height'] = df_clean['Height'].apply(parse_height)

8- Weight Cleaning

In [48]:
def parse_weight(w):
    try:
        if pd.isna(w): return np.nan
        if isinstance(w, (int, float)): return float(w)
        w_str = str(w).strip()
        if 'lbs' in w_str: return round(float(w_str.replace('lbs', '')) * 0.453592, 1)
        if 'kg' in w_str: return float(w_str.replace('kg', ''))
        return float(w_str)
    except: return np.nan

df_clean['Weight'] = df_clean['Weight'].apply(parse_weight)

9- Currency cleaning (Value, Wage, Release Clause) 

In [49]:
def parse_currency(val):
    try:
        if pd.isna(val): return 0
        if isinstance(val, (int, float)): return float(val)
        val_str = str(val).replace('€', '').strip()
        if 'M' in val_str: return float(val_str.replace('M', '')) * 1_000_000
        if 'K' in val_str: return float(val_str.replace('K', '')) * 1_000
        return float(val_str) if val_str else 0
    except: return 0

for col in ['Value', 'Wage', 'Release Clause']:
    df_clean[col] = df_clean[col].apply(parse_currency)

10- Removing star from Rating column

In [50]:
def clean_stars(val):
    try:
        if pd.isna(val): return np.nan
        return int(str(val).replace('★', '').strip())
    except: return np.nan

for col in ['W/F', 'SM', 'IR']:
    df_clean[col] = df_clean[col].apply(clean_stars)

11- Cleaning Hits

In [51]:
df_clean['Hits'] = df_clean['Hits'].astype(str).str.replace('\n', '').str.strip()

df_clean['Hits'] = df_clean['Hits'].apply(
    lambda x: float(x.replace('K','')) * 1000 if 'K' in x 
    else float(x) if x.replace('.','',1).isdigit() 
    else None
).astype('Int64')

In [52]:
# team contract
def extract_team_info(text):
    try:
        if pd.isna(text): return 'Free Agent', np.nan, np.nan
        parts = [p.strip() for p in str(text).split('\n') if p.strip()]
        team = 'Unknown'
        start, end = np.nan, np.nan
        
        for part in parts:
            if '~' not in part and len(part) > 2 and not part.isdigit():
                team = part
                break
        
        for part in parts:
            if '~' in part:
                years = re.findall(r'\d{4}', part)
                if len(years) >= 2:
                    start, end = int(years[0]), int(years[1])
                break
        
        return team, start, end
    except: return 'Unknown', np.nan, np.nan

team_info = df_clean['Team & Contract'].apply(extract_team_info)
df_clean['Team'] = [x[0] for x in team_info]
df_clean['Contract_Start'] = [x[1] for x in team_info]
df_clean['Contract_End'] = [x[2] for x in team_info]
print(f"Team info: extracted {df_clean['Team'].nunique()} teams")

Team info: extracted 714 teams


12-Player Position Categorization

In [53]:
# Postion
df_clean['Primary_Position'] = df_clean['Positions'].str.split().str[0]

position_map = {
    'GK': 'GK',
    'CB': 'DEF', 'LB': 'DEF', 'RB': 'DEF', 'LWB': 'DEF', 'RWB': 'DEF',
    'CDM': 'MID', 'CM': 'MID', 'CAM': 'MID', 'LM': 'MID', 'RM': 'MID',
    'LW': 'ATT', 'RW': 'ATT', 'ST': 'ATT', 'CF': 'ATT'
}
df_clean['Position_Category'] = df_clean['Primary_Position'].map(position_map)
print(f"Positions: {df_clean['Position_Category'].value_counts().to_dict()}")

Positions: {'MID': 7029, 'DEF': 6247, 'ATT': 3628, 'GK': 2075}


13- Loan Status

In [54]:
# Loan status
df_clean['On_Loan'] = df_clean['Loan Date End'].notna().astype(int)
print(f"Loan status: {df_clean['On_Loan'].sum()} players on loan")

Loan status: 1013 players on loan


14-Drop Unnecessary Columns

In [55]:
df_clean.drop(columns=['Team & Contract', 'Positions', 'Loan Date End'], inplace=True)

15- Handling Missing values

In [56]:
# Fill numerical with median
num_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Fill categorical with Unknown
cat_cols = df_clean.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()
        fill_val = mode_val[0] if len(mode_val) > 0 else 'Unknown'
        df_clean[col].fillna(fill_val, inplace=True)

print(f"Missing values after handling: {df_clean.isnull().sum().sum()}")

Missing values after handling: 0


16- Handling Outliers

In [57]:
def cap_outliers_iqr(data, col, multiplier=2.0):
    Q1, Q3 = data[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - multiplier * IQR, Q3 + multiplier * IQR
    data[col] = np.clip(data[col], lower, upper)
    return data
for col in ['Height', 'Weight']:
    df_clean = cap_outliers_iqr(df_clean, col, multiplier=2.5)
    

17- Future Engineering

In [58]:
# Age category
df_clean['Age_Category'] = pd.cut(df_clean['Age'], bins=[0, 21, 27, 32, 50], 
                                   labels=['Young', 'Prime', 'Experienced', 'Veteran'])

# Performance indices
df_clean['Attack_Score'] = (df_clean['PAC'] + df_clean['SHO'] + df_clean['DRI']) / 3
df_clean['Defense_Score'] = df_clean['DEF']
df_clean['Passing_Score'] = df_clean['PAS']
df_clean['Physical_Score'] = df_clean['PHY']

# Growth metrics
df_clean['Potential_Growth'] = df_clean['POT'] - df_clean['OVA']
df_clean['Value_per_Rating'] = df_clean['Value'] / (df_clean['OVA'] + 1)

# BMI
df_clean['BMI'] = df_clean['Weight'] / ((df_clean['Height'] / 100) ** 2)

# Contract
df_clean['Contract_Length'] = df_clean['Contract_End'] - df_clean['Contract_Start']
df_clean['Years_Remaining'] = df_clean['Contract_End'] - 2021

# Star players
df_clean['Is_Star'] = (df_clean['OVA'] >= 85).astype(int)


18- Encoding

In [59]:
# Binary: foot
df_clean['foot_encoded'] = (df_clean['foot'] == 'Right').astype(int)

# Ordinal: Work rates
work_rate_map = {'Low': 1, 'Medium': 2, 'High': 3}
df_clean['AttackRate'] = df_clean['A/W'].map(work_rate_map)
df_clean['DefenseRate'] = df_clean['D/W'].map(work_rate_map)

# One-Hot: Position_Category
pos_dummies = pd.get_dummies(df_clean['Position_Category'], prefix='Pos')
df_clean = pd.concat([df_clean, pos_dummies], axis=1)

# Frequency Encoding: Nationality & Team
df_clean['Nationality_freq'] = df_clean['Nationality'].map(
    df_clean['Nationality'].value_counts() / len(df_clean))
df_clean['Team_freq'] = df_clean['Team'].map(
    df_clean['Team'].value_counts() / len(df_clean))

# Label Encoding(Primary_Position)
le = LabelEncoder()
df_clean['Position_encoded'] = le.fit_transform(df_clean['Primary_Position'])

19- Saved Processed Data

In [60]:
df.to_csv(r"D:\Regression.ml\Project-ML\data\processed\fifa21_clean.csv", index=False)
